# Financial Sentiment Analysis - Model Fine-Tuning

This notebook fine-tunes Llama-3 8B on financial sentiment data using Unsloth + QLoRA.

**Expected Runtime**: 30-40 minutes on T4 GPU

**Steps**:
1. Install dependencies
2. Load base model
3. Prepare datasets
4. Add LoRA adapters
5. Train the model
6. Export to GGUF format
7. Download for local use

## Step 1: Install Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

## Step 2: Load Base Model (4-bit Quantized)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 512
dtype = None  # Auto-detect
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("✓ Base model loaded successfully!")

## Step 3: Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("✓ LoRA adapters attached!")

## Step 4: Prepare Training Data

In [ ]:
from datasets import load_dataset, concatenate_datasets
import pandas as pd

# Instruction template
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

instruction = "Analyze the sentiment of this financial headline. Output one word: Positive, Negative, or Neutral."

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_text, output) + tokenizer.eos_token
        texts.append(text)
    return {"text": texts}

print("Loading datasets...")

# 1. Financial PhraseBank
phrasebank = load_dataset("financial_phrasebank", "sentences_allagree", trust_remote_code=True)
label_map = {0: "Negative", 1: "Neutral", 2: "Positive"}

phrasebank_formatted = []
for item in phrasebank['train']:
    phrasebank_formatted.append({
        "instruction": instruction,
        "input": item['sentence'],
        "output": label_map[item['label']]
    })

# 2. Twitter Financial News
twitter = load_dataset("zeroshot/twitter-financial-news-sentiment", trust_remote_code=True)
twitter_label_map = {0: "Negative", 1: "Positive", 2: "Neutral"}  # Bearish->Negative, Bullish->Positive

twitter_formatted = []
for item in twitter['train']:
    twitter_formatted.append({
        "instruction": instruction,
        "input": item['text'],
        "output": twitter_label_map[item['label']]
    })

# Combine datasets
all_data = phrasebank_formatted + twitter_formatted
print(f"Total examples: {len(all_data)}")

# Convert to HuggingFace Dataset
from datasets import Dataset
dataset = Dataset.from_pandas(pd.DataFrame(all_data))

# Split into train/val (90/10)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset['train']
val_dataset = dataset['test']

print(f"Train: {len(train_dataset)}")
print(f"Val: {len(val_dataset)}")

# Apply formatting
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)

print("\n✓ Datasets prepared!")
print("\nSample:")
print(train_dataset[0]['text'][:300])

## Step 5: Configure Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        load_best_model_at_end=True,
    ),
)

print("✓ Trainer configured!")

## Step 6: Start Training (20-40 minutes)

In [ ]:
print("Starting training...")
print("This will take approximately 20-40 minutes on T4 GPU")
print("="*60)

trainer_stats = trainer.train()

print("\n" + "="*60)
print("✓ Training complete!")
print(f"Final loss: {trainer_stats.training_loss:.4f}")

## Step 7: Test the Model

In [ ]:
# Enable fast inference
FastLanguageModel.for_inference(model)

# Test examples
test_cases = [
    "The company reported record profits this quarter.",
    "Stock prices plummeted after the earnings miss.",
    "Operating costs increased, but margins expanded.",
    "The CEO resigned unexpectedly.",
    "Quarterly revenue met analyst expectations."
]

print("Testing model predictions:")
print("="*60)

for text in test_cases:
    inputs = tokenizer(
        [
            alpaca_prompt.format(
                instruction,
                text,
                ""
            )
        ], return_tensors="pt").to("cuda")
    
    outputs = model.generate(**inputs, max_new_tokens=10, use_cache=True)
    result = tokenizer.batch_decode(outputs)[0]
    
    # Extract the response
    response = result.split("### Response:")[-1].split("<|end_of_text|>")[0].strip()
    
    print(f"\nInput: {text}")
    print(f"Prediction: {response}")

## Step 8: Export to GGUF Format

In [ ]:
print("Merging LoRA adapters...")
model.save_pretrained_merged("financial-sentiment-merged", tokenizer, save_method="merged_16bit")

print("Converting to GGUF format...")
model.save_pretrained_gguf("financial-sentiment", tokenizer, quantization_method="q4_k_m")

print("\n✓ Model exported successfully!")
print("\nFiles created:")
!ls -lh financial-sentiment*

## Step 9: Download the Model

In [ ]:
from google.colab import files

# The GGUF file will be named something like:
# financial-sentiment-Q4_K_M.gguf

print("Available GGUF files:")
!ls -lh *.gguf

print("\nStarting download...")
print("This file is ~4.8GB, download may take several minutes")

# Find the GGUF file
import glob
gguf_files = glob.glob("*.gguf")
if gguf_files:
    gguf_file = gguf_files[0]
    print(f"Downloading: {gguf_file}")
    files.download(gguf_file)
    print("\n✓ Download complete!")
    print(f"\nRename the downloaded file to: financial-sentiment-model.gguf")
else:
    print("❌ No GGUF file found!")

## Next Steps

1. **Rename** the downloaded file to `financial-sentiment-model.gguf`
2. **Move** it to `backend/sentiment-service/` directory
3. **Import** to Ollama:
   ```bash
   cd backend/sentiment-service
   ollama create financial-sentiment -f Modelfile
   ```
4. **Test** the model:
   ```bash
   ollama run financial-sentiment "The company beat earnings expectations."
   ```
5. **Start** the sentiment service:
   ```bash
   python app.py
   ```

See `TRAINING_GUIDE.md` for detailed instructions!